# Introduction to Lasso Regression and Ames Housing Dataset

## The Dataset: Ames Housing
The Ames Housing dataset is a comprehensive alternative to the classic Boston Housing dataset. It contains 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa. The target variable we want to predict is the `SalePrice`. With its high dimensionality and a rich mix of numerical and categorical features, it serves as an excellent benchmark for data cleaning, feature engineering, and advanced regression techniques.

## The Model: Lasso Regression
Lasso (Least Absolute Shrinkage and Selection Operator) is a linear regression model that incorporates **L1 regularization**. 
Unlike standard ordinary least squares (OLS) regression, Lasso applies a penalty equal to the absolute value of the coefficients' magnitude. This key mechanism allows Lasso to perform **automatic feature selection** by shrinking the weights of less relevant features exactly to zero. As a result, it yields a simpler, more robust, and highly interpretable model, which is especially useful when dealing with datasets that have a large number of features.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

In [ ]:
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df = ames.frame
df.columns

## Data Cleaning: Intelligent Missing Value Imputation

In this section, we handle missing data (`NaN`) using three distinct strategies tailored to the specific context of the Ames dataset:

1. **Context-Aware Imputation ("None"):** For certain structural features (e.g., `PoolQC`, `GarageType`, `BsmtQual`), a missing value is not an error but indicates the physical absence of that amenity (e.g., the house has no pool or no basement). We iterate through these specific columns and replace `NaN` with the string `"None"`. For pandas `category` data types, we explicitly add `"None"` to the allowed categories first.
2. **Numerical Imputation (Median):** For all numeric columns (integers and floats), missing values are filled using the median of the respective column. The median is preferred over the mean as it is more robust to outliers (e.g., extremely large houses).
3. **Categorical Imputation (Mode):** For any remaining categorical or text-based features where a missing value truly represents missing data, we fill the gaps using the mode (the most frequent value) of that specific column.

In [ ]:
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

for col in cols_none:
    if col in df.columns:
        if df[col].dtype.name == 'category':
            df[col] = df[col].cat.add_categories("None")
        df[col] = df[col].fillna("None")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object', 'category']).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

## Feature Engineering and One-Hot Encoding

In this step, we prepare the dataset for the machine learning model by creating new features and transforming categorical variables into a numerical format:

1. **Feature Creation:** We engineer a new aggregate feature, `TotalSF`, by summing the square footage of the basement, first floor, and second floor. We also drop the `Id` column, as it is merely a row identifier and carries no predictive power.
2. **One-Hot Encoding (Focus):** Machine learning algorithms require numerical input, so we convert categorical text variables into binary (0 or 1) columns using `pd.get_dummies()`. 
   * **The Dummy Variable Trap:** We explicitly use the `drop_first=True` parameter. This drops one category per categorical feature to avoid perfect multicollinearity (the "dummy variable trap"). For instance, if a house category is either "A" or "B", we only need one column (e.g., "Is_B") to represent both states. Preventing multicollinearity is strictly required for the mathematical stability and interpretability of linear models like Lasso.
3. **Target Separation:** Finally, we isolate our target variable `SalePrice` (`y`) from our predictor features (`X`) and print the column count to show how much the dataset expanded after encoding.

In [ ]:
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

if 'Id' in df.columns:
    df = df.drop('Id', axis=1)

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop('SalePrice', axis=1)
y = df_encoded['SalePrice']

print(f'Cols before transform: {df.shape[1]}')
print(f'Cols after One-Hot Encoding: {X.shape[1]}')

## Data Splitting, Scaling, and Model Training

In this section, we prepare the data for training and fit an optimized Lasso model:

1. **Train-Test Split:** We split our dataset into a training set (80%) and a testing set (20%) to ensure we can evaluate the model's performance on unseen data.
2. **Feature Scaling:** We use `StandardScaler` to standardize our features (mean = 0, variance = 1). 
   * *Crucial Note:* We use `fit_transform()` on the training data but only `transform()` on the testing data. This prevents **data leakage**, ensuring the model learns absolutely nothing about the test set distribution during training. 
   * Scaling is strictly mandatory for Lasso Regression; without it, features with naturally larger scales (e.g., square footage) would be unfairly penalized by the L1 regularization compared to features with smaller scales (e.g., number of rooms).
3. **Model Training with Cross-Validation:** Instead of manually guessing the optimal regularization penalty (`alpha`), we use `LassoCV`. This algorithm automatically performs 5-fold cross-validation (`cv=5`) on the training set to test multiple `alpha` values under the hood, ultimately selecting the one that minimizes the error.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_model = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

## Model Evaluation and Feature Selection

In this final step, we evaluate the model's performance on the unseen test data and examine the effects of Lasso's built-in feature selection:

1. **Making Predictions and Metric Calculation:** We generate price predictions (`y_pred`) for the scaled test set. We evaluate the model using **RMSE** (Root Mean Squared Error), which gives us the average prediction error in actual dollars (USD), and **R²** (R-squared), which represents the proportion of variance in the target variable explained by the model.
2. **Analyzing the Coefficients (Weights):** We extract the learned weights (`lasso_model.coef_`). This highlights the most powerful feature of Lasso Regression (L1 regularization): it acts as an automated feature selector. By counting the non-zero and zero weights, we can see exactly how many features the model deemed important (`kept`) versus how many it completely discarded (`dropped`) to prevent overfitting and reduce complexity.

In [ ]:
y_pred = lasso_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse} USD")
print(f"R^2: {r2}")

In [ ]:
weights = lasso_model.coef_
kept = np.sum(weights != 0)
zerod = np.sum(weights == 0)

print(f"Model got {len(weights)} columns (features)")
print(f"It kept: {kept}")
print(f"It dropped (set weight to 0): {zerod}")

In [ ]:
weights_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Weight': lasso_model.coef_
})

weights_sorted = weights_df.sort_values(by='Weight', ascending=False)

print('---- 10 Features Raising the Price ----')
print(weights_sorted.head(10).to_string(index=False))

print('\n---- 5 Features Decreasing the Price ----')
print(weights_sorted.tail(5).sort_values(by='Weight', ascending=True).to_string(index=False))

## Advanced Evaluation: RMSLE and Residual Plot

In this section, we dive deeper into the model's errors using metrics and visualizations tailored for regression tasks:

1. **RMSLE (Root Mean Squared Logarithmic Error):** This is the official evaluation metric for the Ames dataset on Kaggle. We use `np.clip` to force any negative predictions to zero, preventing math errors since the logarithm of a negative number is undefined. RMSLE is crucial because it measures relative (percentage) error rather than absolute error—meaning a $10,000 mistake on a cheap house is penalized more heavily than the same dollar mistake on a mansion.
2. **Calculating Residuals:** We calculate the raw prediction errors (residuals) by subtracting the predicted prices from the actual actual prices (`y_test - y_pred`).
3. **Visualizing the Errors:** We generate a scatter plot using `seaborn` with the predicted prices on the x-axis and the residuals on the y-axis, adding a red dashed line at zero to represent a perfect prediction.

## Interpreting the Residual Plot Results

Based on the generated plot[cite: 2], we can diagnose how our model is actually behaving across different price ranges:

* **Heteroscedasticity (The Funnel Shape):** Ideally, the points should form a random, uniform cloud around the red zero-line. However, our plot shows a distinct "funnel" shape spreading outwards to the right[cite: 2]. This is known as heteroscedasticity, meaning the variance of the model's errors is not constant.
* **Accuracy on Mid-Range Homes:** For houses predicted between $100,000 and $200,000, the data points are densely clustered close to the red dashed line[cite: 2]. This confirms the Lasso model is quite accurate and reliable for standard, average-priced properties[cite: 2].
* **Systematic Errors on Luxury Real Estate:** As the predicted price moves past $300,000, the spread of the residuals increases dramatically[cite: 2]. Notably, there are severe positive outliers (with residuals hitting 200,000 and even 300,000)[cite: 2]. A positive residual (`Actual - Predicted > 0`) means the actual price was much higher than what the model guessed. This proves that our simple linear model struggles to capture the premium value of luxury homes and systematically underprices them[cite: 2].

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_log_error

y_pred_safe = np.clip(y_pred, a_min=0, a_max=None)
rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred_safe))

print(f"RMSLE: {rmsle:.4f}")

residual = y_test - y_pred

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred, y=residual, alpha=0.6, color='royalblue')

plt.axhline(y=0, color='crimson', linestyle='--', linewidth=2)

plt.title('WResidual Plot - LassoCV')
plt.xlabel('Expected price (USD)')
plt.ylabel('Residual')
plt.tight_layout()
plt.show()

## Overfitting Check and Learning Curves

In this section, we diagnose whether our model is memorizing the data (overfitting) or if it has generalized well, using both static metrics and dynamic learning curves:

1. **Static Train vs. Test Error:** We first calculate the standard RMSE on the data the model saw during training (`X_train_scaled`) and compare it to the unseen test data (`X_test_scaled`). A massive gap between these two numbers would indicate severe overfitting. 
2. **Generating Learning Curves:** We use `scikit-learn`'s `learning_curve` function to train the Lasso model multiple times on increasingly larger subsets of our training data (from 10% to 100% in 10 steps). 
3. **Cross-Validation Scoring:** For each subset size, the function performs 5-fold cross-validation. Since `scikit-learn` maximizes scores by default, we use `neg_root_mean_squared_error` and invert the sign later to get standard positive RMSE values.
4. **Visualization:** We plot the mean training error (blue line) and the mean validation error (orange line) against the number of training samples used. 

## Interpreting the Learning Curve Results

Based on the generated plot and printed metrics[cite: 3], we can draw several important conclusions about our model's current state:

* **Healthy Generalization (No Overfitting):** The printed metrics show a training RMSE of 32,979 USD and a test RMSE of 36,615 USD[cite: 3]. The difference is only 3,636 USD, which proves that Lasso's L1 regularization successfully prevented the model from memorizing the training data[cite: 3]. 
* **Increasing Training Error:** As seen on the blue line, the training error starts very low when the data sample is tiny, but steadily increases as more data is added[cite: 3]. This is completely normal; it is easy for a model to perfectly fit 50 houses, but much harder to draw a single straight line that perfectly fits 800 houses[cite: 3].
* **Model Capacity Reached (High Bias):** The validation error (orange line) remains relatively flat and high (hovering between 35,000 and 41,000 USD) regardless of how much data is fed into it[cite: 3]. Because the validation line does not show a strong downward trend as it moves to the right, providing this specific model with thousands of additional housing records will not significantly improve its accuracy[cite: 3]. To reduce the error further, we need to increase the model's complexity (e.g., add polynomial features or switch to non-linear algorithms like Random Forest).

In [ ]:
from sklearn.model_selection import learning_curve

y_train_pred = lasso_model.predict(X_train_scaled)
y_test_pred = lasso_model.predict(X_test_scaled)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"RMSE Training: {rmse_train:,.0f} USD")
print(f"RMSE Test:    {rmse_test:,.0f} USD")
print(f"Difference:      {rmse_test - rmse_train:,.0f} USD")

train_sizes, train_scores, test_scores = learning_curve(
    estimator=lasso_model,
    X=X_train_scaled,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

train_scores_mean = -train_scores.mean(axis=1)
test_scores_mean = -test_scores.mean(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores_mean, label='Training error', color='royalblue', marker='o')
plt.plot(train_sizes, test_scores_mean, label='Validation error', color='darkorange', marker='s')

plt.title('Learning curve')
plt.xlabel('Amount of training samples')
plt.ylabel('RMSE (USD)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()